# Research Paper Ingestion & Processing

In [4]:
%pip install arxiv langchain langchain_community pypdf


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tempfile
import time

## Fetching Research Papers from ArXiv
- Query ArXiv for academic papers based on a search term.
- Instead of storing PDFs, we load them **temporarily** then extract the text.

In [6]:
def fetch_papers(query="LLM", max_results=5):
    """
    Fetches academic papers from ArXiv based on a search term, processes the PDFs,
    extracts text, and splits the content into smaller, structured chunks.

    Parameters:
        query (str): The search term to query ArXiv for research papers.
        max_results (int): The maximum number of papers to retrieve.

    Returns:
        all_chunks: A list of document chunks extracted from the fetched papers.
    """
    
    # Initialize an ArXiv client
    client = arxiv.Client()

    # Search query with sorting (descending order to get the latest papers)
    search = arxiv.Search(query=query, max_results=max_results, sort_order=arxiv.SortOrder.Descending)
    
    all_chunks = []  # List to store extracted text chunks
    
    for result in client.results(search):
        # Display the title of the paper currently being processed
        print(f"Processing: {result.title}")
        
        # Create a temporary file to store the downloaded PDF
        with tempfile.NamedTemporaryFile(delete=True, suffix=".pdf") as temp_pdf:
            try:
                result.download_pdf(filename=temp_pdf.name)
                print(f"-> Downloaded: {result.title}")
                
                # Load PDF file 
                loader = PyPDFLoader(temp_pdf.name)
                papers = loader.load()
                
                # Combine all pages into a single string
                full_text = " ".join(p.page_content for p in papers)
                
                # Split the extracted text into chunks
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                paper_chunks = text_splitter.create_documents([full_text])
                
                # Add extracted chunks to the list
                all_chunks.extend(paper_chunks) 
                
            except (FileNotFoundError, ConnectionResetError) as e:
                print("Error occurred while downloading or processing PDF:", e)
                time.sleep(5)
    
    # Print total number of extracted chunks
    print(f"Total chunks extracted: {len(all_chunks)}")

    return all_chunks

## Testing the Paper Retrieval Pipeline

In [7]:
query = "Large Language Models"
paper_chunks = fetch_papers(query=query, max_results=3)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")

Processing: Lost in Translation: Large Language Models in Non-English Content Analysis
-> Downloaded: Lost in Translation: Large Language Models in Non-English Content Analysis
Processing: Cedille: A large autoregressive French language model
-> Downloaded: Cedille: A large autoregressive French language model
Processing: How Good are Commercial Large Language Models on African Languages?
-> Downloaded: How Good are Commercial Large Language Models on African Languages?
Total chunks extracted: 480

Chunk 1:
Lost in Translation
May 2023
A report from
Gabriel Nicholas 
Aliya Bhatia
Large Language Models in 
Non-English Content Analysis GABRIEL NICHOLAS
Research Fellow at the Center for Democracy & T echnology.
ALIYA BHATIA
Policy Analyst, Free Expression Project at the Center for 
Democracy & T echnology...

Chunk 2:
civil liberties in the digital age. W e shape technology policy, governance, 
and design with a focus on equity and democratic values. Established in 
1996, CDT has been a t

In [10]:
query = 'Transformers'
paper_chunks = fetch_papers(query=query, max_results=3)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")

Processing: The Xi-transform for conformally flat space-time
-> Downloaded: The Xi-transform for conformally flat space-time
Processing: Multiple basic hypergeometric transformation formulas arising from the balanced duality transformation
-> Downloaded: Multiple basic hypergeometric transformation formulas arising from the balanced duality transformation
Processing: The Fourier and Hilbert transforms under the Bargmann transform
-> Downloaded: The Fourier and Hilbert transforms under the Bargmann transform
Total chunks extracted: 376

Chunk 1:
arXiv:gr-qc/0612006v1  1 Dec 2006
The Ξ-transform for conformally ﬂat
space-time
George A.J. Sparling
Laboratory of Axiomatics
Department of Mathematics
University of Pittsburgh
Pittsburgh, Pennsylvania, 15260, USA
Abstract
The Ξ -transform is a new spinor transform arising naturally in
Einstein’s g...

Chunk 2:
ﬁned transforms: one a two-variable transform on the Lie group
SU(2, C), the other a transform on the space of null split octaves.
The 